# Environment setup (one-time)

Creates the `vrt` conda environment (GDAL, geopandas, rasterio, PyTorch CUDA, ...) and registers it as the **Python (vrt)** Jupyter kernel.

*Only needed once per SageMaker space. Afterwards, select "Python (vrt)" as the kernel for `train.ipynb` and the `pipeline_{STATE}.ipynb` notebooks.*

In [ ]:
# =============================
# VRT ENV BOOTSTRAP (GPU-SAFE)
# =============================

import subprocess

def run(cmd):
    print(f"\n>> {cmd}")
    subprocess.check_call(cmd, shell=True)

# -----------------------------
# 0. clean up
# -----------------------------
run("conda deactivate || true")
run("conda remove -n vrt --all -y || true")
run("conda clean -y --all")

# deterministic solver behavior
run("conda config --set channel_priority strict")

# -----------------------------
# 1. create vrt env (conda ONLY)
# -----------------------------
run("""
conda create -y -n vrt -c conda-forge \
  python=3.11 \
  gdal \
  geopandas \
  rasterio \
  shapely \
  fiona \
  pyproj \
  rtree \
  pyogrio \
  numpy \
  pandas \
  scikit-image \
  tqdm \
  matplotlib \
  pillow \
  ipywidgets \
  requests \
  boto3
""")

# -----------------------------
# 2. install PyTorch CUDA (pip wheels)
# -----------------------------
run("conda run -n vrt python -m pip install --upgrade pip wheel setuptools")

run("""
conda run -n vrt python -m pip install \
  torch torchvision \
  --index-url https://download.pytorch.org/whl/cu121
""")

# -----------------------------
# 3. register Jupyter kernel
# -----------------------------
run("conda run -n vrt python -m pip install ipykernel")

run("""
conda run -n vrt python -m ipykernel install \
  --user \
  --name vrt \
  --display-name "Python (vrt)"
""")

print("\nEnvironment 'vrt' created.")
print("Restart Jupyter and switch kernel to 'Python (vrt)'")